# Modeling

In [8]:
import os, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, make_scorer, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

In [2]:
DATA_DIR = r"D:\Ameng\Data Science Project\heart-failure-prediction\data"
df = pd.read_csv(os.path.join(DATA_DIR, "heart_failure_clinical_records_dataset.csv"))

In [3]:
X = df.drop("DEATH_EVENT", axis=1)
y = df["DEATH_EVENT"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (299, 12)
y shape: (299,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (239, 12)
X_test: (60, 12)


In [6]:
print("Train target distribution:")
print(y_train.value_counts())

print("\nTrain target proportion:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts())

print("\nTest target proportion:")
print(y_test.value_counts(normalize=True))

Train target distribution:
DEATH_EVENT
0    162
1     77
Name: count, dtype: int64

Train target proportion:
DEATH_EVENT
0    0.677824
1    0.322176
Name: proportion, dtype: float64

Test target distribution:
DEATH_EVENT
0    41
1    19
Name: count, dtype: int64

Test target proportion:
DEATH_EVENT
0    0.683333
1    0.316667
Name: proportion, dtype: float64


In [9]:
def build_preprocessor():
    log_cols = ["creatinine_phosphokinase", "serum_creatinine", "platelets", "time"]
    scale_cols = ["age", "ejection_fraction", "serum_sodium"]
    bin_cols = ["anaemia", "diabetes", "high_blood_pressure", "sex", "smoking"]

    log_pipeline = Pipeline([
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler())
    ])
    
    scale_pipeline = Pipeline([
        ("scale", StandardScaler())
    ])

    return ColumnTransformer([
        ("log", log_pipeline, log_cols),
        ("scale", scale_pipeline, scale_cols),
        ("pass", "passthrough", bin_cols)
    ])

## Cross Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scorers = {
    "AUC": make_scorer(roc_auc_score, response_method="predict_proba"),
    "F1": make_scorer(f1_score),
    "PRE": make_scorer(precision_score),
    "REC": make_scorer(recall_score),
}

def cv_table(X, y, models):
    rows = []
    for name, model in models.items():
        r = {m: cross_val_score(model, X, y, cv=cv, scoring=s).mean().round(4)
             for m, s in scorers.items()}
        r |= {"model": name}; rows.append(r)
    return pd.DataFrame(rows).set_index("model")

In [ ]:
models = {
    "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
    "SVM": CalibratedClassifierCV(SVC(kernel="rbf", class_weight="balanced"), ensemble=False),
    "kNN": KNeighborsClassifier(n_neighbors=5),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
}
result = cv_table(X_train, y_train, models)
print(result.sort_values("AUC", ascending=False))

Based on 5-fold cross-validation result, the top three models (XGBoost, Random Forest, and Logistic Regression) will be selected for hyperparameter tuning.

## Hyperparameter Tuning

In [ ]:
param_lr = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "l1_ratio": [0, 1]
}

param_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

param_xgb = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

In [ ]:
grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight="balanced", solver="liblinear"),
    param_grid=param_lr,
    cv=cv, scoring="roc_auc", n_jobs=-1)
grid_lr.fit(X_train, y_train)
print("Best param:", grid_lr.best_params_)
print("Best CV AUC:", grid_lr.best_score_.round(4))

In [ ]:
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced"),
    param_grid=param_rf,
    cv=cv, scoring="roc_auc", n_jobs=-1)

grid_rf.fit(X_train, y_train)

print("Best param:", grid_rf.best_params_)
print("Best CV AUC:", grid_rf.best_score_.round(4))

In [ ]:
grid_xgb = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric="logloss"),
    param_grid=param_xgb,
    cv=cv, scoring="roc_auc", n_jobs=-1
)

grid_xgb.fit(X_train, y_train)

print("Best param:", grid_xgb.best_params_)
print("Best CV AUC:", grid_xgb.best_score_.round(4))

## Tuned Model Comparison

In [ ]:
final_models = {
    "LogReg": grid_lr.best_estimator_,
    "RandomForest": grid_rf.best_estimator_,
    "XGBoost": grid_xgb.best_estimator_
}
res_final = cv_table(X_train, y_train, final_models)
print(res_final.sort_values("AUC", ascending=False))

Based on hyperparameter tuning results using 5-fold cross-validation, XGBoost was selected as the best model candidate because it yielded the highest AUC (0.9218) <br>
and Precision (0.8079), with an F1-score (0.7638) close to that of Logistic Regression (0.7706). Although Logistic Regression achieved the highest Recall (0.8042), XGBoost <br>
delivered superior overall performance based on the combination of evaluation metrics.

In [ ]:
best_model = grid_xgb.best_estimator_

print("Best parameters:", grid_xgb.best_params_)
print("Best CV AUC:", round(grid_xgb.best_score_, 4))

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

test_auc = roc_auc_score(y_test, y_proba)
test_f1 = f1_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_recall = recall_score(y_test, y_pred)

print(f"Test AUC: {test_auc:.4f}")
print(f"Test F1: {test_f1:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")

## Threshold Optimization
F1, Precision, and Recall depend on the decision cut-off. With imbalanced data (68:32), the default threshold of 0.5 is rarely optimal. <br>
The threshold is selected based on out-of-fold (OOF) CV, not the test set, to ensure the cut-off selection is unbiased. AUC is unaffected by the threshold.

In [ ]:
oof = cross_val_predict(best_model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

thresholds = np.arange(0.10, 0.90, 0.01)
f1_by_t = [f1_score(y_train, oof >= t) for t in thresholds]
best_t = thresholds[np.argmax(f1_by_t)]

plt.plot(thresholds, f1_by_t)
plt.axvline(best_t, color="red", ls="--", label=f"best = {best_t:.2f}")
plt.xlabel("Threshold")
plt.ylabel("OOF CV F1")
plt.legend()
plt.title("F1 vs Threshold (OOF CV)")
plt.show()

print("Best threshold:", round(best_t, 2))
print("OOF CV F1 best threshold:", round(max(f1_by_t), 4))

In [ ]:
y_pred_default = (y_proba >= 0.50).astype(int)
y_pred_opt = (y_proba >= best_t).astype(int)

rows = []
for name, yp in [("default (0.5)", y_pred_default),
                 (f"optimized ({best_t:.2f})", y_pred_opt)]:
    rows.append({"threshold": name,
                 "F1": f1_score(y_test, yp),
                 "PRE": precision_score(y_test, yp),
                 "REC": recall_score(y_test, yp)})

print(pd.DataFrame(rows).round(4))
print("Test AUC (threshold-independent):", round(test_auc, 4))

## Results Visualization

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_opt,
    display_labels=["Survived", "Death"])
plt.title(f"Confusion Matrix (threshold = {best_t:.2f})")
plt.show()

The confusion matrix at a threshold of 0.33 shows 38 True Negatives, 13 True Positives, 3 False Positives, and 6 False Negatives. The model successfully detected <br>
13 out of 19 Death cases (Recall: 68.42%). Adjusting the threshold improved the model's ability to detect Death cases compared to the default threshold of 0.5, although 6 <br>
False Negative cases remained.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.plot(fpr, tpr, label=f"XGBoost (AUC = {test_auc:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.title("ROC Curve (Test Set)")
plt.show()

The XGBoost ROC curve on the test set yielded an AUC of 0.879, indicating good discrimination capability in distinguishing between the 'Survived' and 'Death' classes. <br>
The curve lies well above the random classifier line, demonstrating that the model performs better than random prediction.

In [ ]:
imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values()
imp.plot(kind="barh", figsize=(9, 5))
plt.title("XGBoost Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

XGBoost feature importance indicates that `time` is the most dominant feature in the prediction, followed by `serum_creatinine` and `ejection_fraction`. `Age` also makes a <br>
significant contribution, whereas `diabetes` and `high_blood_pressure` have relatively low importance. Feature importance reflects the extent of a feature's contribution <br>
to the model but does not indicate the direction of its influence on the prediction.

## Export Model Result

In [ ]:
MODELS_DIR = r"D:\Ameng\Data Science Project\heart-failure-prediction\models"
os.makedirs(MODELS_DIR, exist_ok=True)
joblib.dump(best_model, os.path.join(MODELS_DIR, "model.joblib"))

metrics = pd.DataFrame([
    {"metric": "AUC", "value": test_auc},
    {"metric": "F1 (optimized)", "value": f1_score(y_test, y_pred_opt)},
    {"metric": "Precision (optimized)", "value": precision_score(y_test, y_pred_opt)},
    {"metric": "Recall (optimized)", "value": recall_score(y_test, y_pred_opt)},
    {"metric": "best_threshold", "value": best_t},
])
metrics.to_csv(os.path.join(MODELS_DIR, "metrics_test.csv"), index=False)
print("Model & metrics saved to:", MODELS_DIR)